# Uncertainty and Multi-Paradigm Benchmark

Notebooks 03 and 04 established that physics-informed detectors can reach 0.89 AUROC using the learned risk map. This notebook adds model-level uncertainty methods (MC dropout, deep ensembles, TTA), tries OOD and spectral approaches, and produces the full 22-detector comparison.

The key question: do model-level uncertainty methods catch hallucinations that mask-level methods (multi-mask, NB04) miss?

## Setup
Requires checkpoints from NB02 (`unet_4x_v2_best.pt`) and NB04 (`risk_map_v1.pt`). Deep ensemble training needs ~3.5 hours on L4.

In [ ]:
!pip install fastmri h5py scikit-image pyyaml tqdm -q

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, h5py, os, glob, json, time
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.ndimage import uniform_filter, sobel, laplace, gaussian_filter

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from google.colab import drive
import subprocess
drive.mount('/content/drive')

val_dst = '/content/data/singlecoil_val/'
os.makedirs(val_dst, exist_ok=True)
subprocess.run(['rsync', '-a', '--ignore-existing',
                '/content/drive/MyDrive/fastmri/singlecoil_val/', val_dst], capture_output=True)
n_files = len(glob.glob(os.path.join(val_dst, '*.h5')))
print(f"Val volumes: {n_files}")

## 1. Models and Utilities

All FFT functions, masks, U-Net, SmallUNet (learned risk), lift procedures, null-space decomposition, evaluation metrics, and NB04 detector functions. Self-contained so the notebook runs independently.

In [ ]:
# ---- FFT ----
def to_kspace(img):
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(img, dim=(-2,-1)), norm='ortho'), dim=(-2,-1))

def from_kspace(ksp):
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(ksp, dim=(-2,-1)), norm='ortho'), dim=(-2,-1))

def center_crop(x, shape):
    h, w = x.shape[-2:]
    th, tw = shape
    return x[..., (h-th)//2:(h-th)//2+th, (w-tw)//2:(w-tw)//2+tw]

# ---- Masks ----
def create_mask(shape, acceleration, mask_type='random', center_fraction=0.08, seed=42):
    H, W = shape
    rng = np.random.RandomState(seed)
    nc = max(1, int(W * center_fraction))
    nt = max(nc + 1, int(W / acceleration))
    no = nt - nc
    mask = np.zeros(W, dtype=np.float32)
    cs = (W - nc) // 2
    mask[cs:cs+nc] = 1
    outer = np.where(mask == 0)[0]
    if mask_type == 'random':
        chosen = rng.choice(outer, size=min(no, len(outer)), replace=False)
        mask[chosen] = 1
    elif mask_type == 'equispaced':
        step = max(1, len(outer) // max(1, no))
        mask[outer[::step][:no]] = 1
    elif mask_type == 'gaussian':
        probs = np.exp(-0.5 * ((outer - W/2) / (W/6))**2); probs /= probs.sum()
        mask[rng.choice(outer, size=min(no, len(outer)), replace=False, p=probs)] = 1
    elif mask_type == 'poisson_disc':
        probs = 1.0 / (1.0 + np.abs(outer - W/2) / (W/4)); probs /= probs.sum()
        mask[rng.choice(outer, size=min(no, len(outer)), replace=False, p=probs)] = 1
    return torch.from_numpy(mask.reshape(1, W))

# ---- U-Net ----
class ConvBlock(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ic, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(True),
            nn.Conv2d(oc, oc, 3, padding=1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(True))
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, channels=(32,64,128,256), dropout_p=0.05):
        super().__init__()
        self.encoders = nn.ModuleList(); self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList(); self.upconvs = nn.ModuleList()
        self.dropout = nn.Dropout2d(p=dropout_p)
        ic = 1
        for ch in channels:
            self.encoders.append(ConvBlock(ic, ch)); self.pools.append(nn.MaxPool2d(2)); ic = ch
        self.bottleneck = ConvBlock(channels[-1], channels[-1]*2)
        for ch in reversed(channels):
            self.upconvs.append(nn.ConvTranspose2d(ch*2, ch, 2, stride=2))
            self.decoders.append(ConvBlock(ch*2, ch))
        self.final = nn.Conv2d(channels[0], 1, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x); x = self.dropout(x)
        x = self.bottleneck(x)
        for up, dec, sk in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape != sk.shape: x = F.pad(x, [0, sk.shape[3]-x.shape[3], 0, sk.shape[2]-x.shape[2]])
            x = torch.cat([x, sk], 1); x = dec(x); x = self.dropout(x)
        return self.final(x)

# ---- SmallUNet (risk map) ----
class SmallUNet(nn.Module):
    def __init__(self, in_ch=2, channels=(32,64,128)):
        super().__init__()
        c1, c2, c3 = channels
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch,c1,3,padding=1),nn.ReLU(True),nn.Conv2d(c1,c1,3,padding=1),nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(c1,c2,3,padding=1),nn.ReLU(True),nn.Conv2d(c2,c2,3,padding=1),nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(c2,c3,3,padding=1),nn.ReLU(True),nn.Conv2d(c3,c3,3,padding=1),nn.ReLU(True))
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(c3,c3*2,3,padding=1),nn.ReLU(True),nn.Conv2d(c3*2,c3*2,3,padding=1),nn.ReLU(True))
        self.up3 = nn.ConvTranspose2d(c3*2,c3,2,stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(c3*2,c3,3,padding=1),nn.ReLU(True),nn.Conv2d(c3,c3,3,padding=1),nn.ReLU(True))
        self.up2 = nn.ConvTranspose2d(c3,c2,2,stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(c2*2,c2,3,padding=1),nn.ReLU(True),nn.Conv2d(c2,c2,3,padding=1),nn.ReLU(True))
        self.up1 = nn.ConvTranspose2d(c2,c1,2,stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(c1*2,c1,3,padding=1),nn.ReLU(True),nn.Conv2d(c1,c1,3,padding=1),nn.ReLU(True))
        self.final = nn.Conv2d(c1,1,1)

    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1)); e3=self.enc3(self.pool(e2))
        b=self.bottleneck(self.pool(e3))
        d3=self.up3(b)
        if d3.shape!=e3.shape: d3=F.pad(d3,[0,e3.shape[3]-d3.shape[3],0,e3.shape[2]-d3.shape[2]])
        d3=self.dec3(torch.cat([d3,e3],1))
        d2=self.up2(d3)
        if d2.shape!=e2.shape: d2=F.pad(d2,[0,e2.shape[3]-d2.shape[3],0,e2.shape[2]-d2.shape[2]])
        d2=self.dec2(torch.cat([d2,e2],1))
        d1=self.up1(d2)
        if d1.shape!=e1.shape: d1=F.pad(d1,[0,e1.shape[3]-d1.shape[3],0,e1.shape[2]-d1.shape[2]])
        d1=self.dec1(torch.cat([d1,e1],1))
        return torch.sigmoid(self.final(d1))

# ---- FastMRI Dataset (for ensemble training) ----
class FastMRIDataset(torch.utils.data.Dataset):
    def __init__(self, root, acceleration=4, mask_type='random', center_fraction=0.08,
                 fixed_masks=True, seed=42, augment=False):
        self.acceleration, self.mask_type = acceleration, mask_type
        self.center_fraction, self.fixed_masks = center_fraction, fixed_masks
        self.seed, self.augment = seed, augment
        self.slices = []
        for h5 in sorted(glob.glob(os.path.join(root, '*.h5'))):
            try:
                with h5py.File(h5, 'r') as f:
                    for s in range(f['kspace'].shape[0]):
                        self.slices.append((h5, s))
            except: pass

    def __len__(self): return len(self.slices)

    def __getitem__(self, idx):
        h5_path, si = self.slices[idx]
        with h5py.File(h5_path, 'r') as f:
            kspace = torch.from_numpy(f['kspace'][si].copy())
            target = torch.from_numpy(f['reconstruction_esc'][si].copy())
        H, W = kspace.shape
        seed = self.seed + idx if self.fixed_masks else self.seed + idx + int(torch.randint(0,1000000,(1,)))
        mask = create_mask((H,W), self.acceleration, self.mask_type, self.center_fraction, seed)
        ifft = center_crop(from_kspace(kspace * mask).abs(), (320,320))
        inp = ((ifft - ifft.min()) / (ifft.max() - ifft.min() + 1e-8)).unsqueeze(0).reshape(1,320,320)
        tgt = ((target - target.min()) / (target.max() - target.min() + 1e-8)).unsqueeze(0).reshape(1,320,320)
        if self.augment and torch.rand(1).item() > 0.5:
            inp = torch.flip(inp, [-1]); tgt = torch.flip(tgt, [-1])
        return inp, tgt

# ---- Load checkpoints (remount if needed) ----
ckpt_dir = '/content/drive/MyDrive/fastmri/checkpoints/'

try:
    ckpt = torch.load(os.path.join(ckpt_dir, 'unet_4x_v2_best.pt'), map_location=device, weights_only=False)
except (FileNotFoundError, OSError):
    print("Drive disconnected. Remounting...")
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    ckpt = torch.load(os.path.join(ckpt_dir, 'unet_4x_v2_best.pt'), map_location=device, weights_only=False)

model = UNet(channels=(32,64,128,256), dropout_p=0.05).to(device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f"UNet: {sum(p.numel() for p in model.parameters()):,} params, epoch {ckpt.get('epoch','?')}")

risk_model = SmallUNet(in_ch=2, channels=(32,64,128)).to(device)
risk_ckpt = torch.load(os.path.join(ckpt_dir, 'risk_map_v1.pt'), map_location=device, weights_only=False)
risk_model.load_state_dict(risk_ckpt['model_state_dict']); risk_model.eval()
print(f"SmallUNet: {sum(p.numel() for p in risk_model.parameters()):,} params")

# ---- NB03 functions ----
def center_embed(crop, full_shape, fill=None):
    H, W = full_shape; th, tw = crop.shape[-2:]
    out = fill.clone() if fill is not None else torch.zeros(*crop.shape[:-2], H, W, dtype=crop.dtype, device=crop.device)
    out[..., (H-th)//2:(H-th)//2+th, (W-tw)//2:(W-tw)//2+tw] = crop
    return out

def lift_unet_to_native(unet_01, esc, ifft_c, gt_c):
    mag = unet_01 * (esc.max()-esc.min()) + esc.min()
    mag_n = center_embed(mag, gt_c.shape[-2:], fill=gt_c.abs())
    return mag_n * torch.exp(1j * torch.angle(ifft_c))

def lift_unet_to_native_inference(unet_01, ifft_mag, ifft_c):
    mn, mx = ifft_mag.min(), ifft_mag.max()
    mag = unet_01 * (mx - mn + 1e-8) + mn
    mag_n = center_embed(mag, ifft_c.shape[-2:], fill=ifft_c.abs())
    return mag_n * torch.exp(1j * torch.angle(ifft_c))

def null_space_decomposition_native(recon, gt, mask, crop_shape=(320,320)):
    error = recon - gt
    error_k = to_kspace(error)
    m = mask.squeeze()
    if m.dim() == 1: m = m.unsqueeze(0).expand(error_k.shape[-2], -1)
    meas_err = from_kspace(error_k * m); null_err = from_kspace(error_k * (1 - m))
    mc = center_crop(meas_err.abs(), crop_shape); nc = center_crop(null_err.abs(), crop_shape)
    me, ne = (mc**2).sum().item(), (nc**2).sum().item()
    total = me + ne + 1e-12
    return {'null_ratio': ne/total, 'meas_ratio': me/total, 'null_map': nc, 'meas_map': mc}

def compute_psf_1d(mask):
    m = mask.squeeze().to(torch.complex64)
    psf = torch.fft.fftshift(torch.fft.ifft(torch.fft.ifftshift(m))).abs()
    return psf / (psf.max() + 1e-12)

def make_psf_2d(mask, size=320):
    psf = compute_psf_1d(mask)
    W = psf.shape[0]
    if W > size: psf = psf[(W-size)//2:(W-size)//2+size]
    elif W < size: psf = F.pad(psf, ((size-W)//2, size-W-(size-W)//2))
    return psf.unsqueeze(0).expand(size, -1)

# ---- Evaluation ----
def compute_auroc_patches(risk, gt, patch_size=32):
    r = risk.detach().cpu().numpy().squeeze() if isinstance(risk, torch.Tensor) else np.asarray(risk).squeeze()
    g = gt.detach().cpu().numpy().squeeze() if isinstance(gt, torch.Tensor) else np.asarray(gt).squeeze()
    H, W = r.shape[-2:]
    rp, gp = [], []
    for i in range(0, H-patch_size+1, patch_size):
        for j in range(0, W-patch_size+1, patch_size):
            rp.append(r[i:i+patch_size, j:j+patch_size].mean())
            gp.append((g[i:i+patch_size, j:j+patch_size]**2).mean())
    ra, ga = np.array(rp), np.array(gp)
    gb = (ga > np.percentile(ga, 80)).astype(float)
    if gb.sum() == 0 or gb.sum() == len(gb): return float('nan')
    return roc_auc_score(gb, ra)

def compute_fpr_at_tpr_patches(risk, gt, tpr=0.95, ps=32):
    r = risk.detach().cpu().numpy().squeeze() if isinstance(risk, torch.Tensor) else np.asarray(risk).squeeze()
    g = gt.detach().cpu().numpy().squeeze() if isinstance(gt, torch.Tensor) else np.asarray(gt).squeeze()
    H, W = r.shape[-2:]
    rp, gp = [], []
    for i in range(0, H-ps+1, ps):
        for j in range(0, W-ps+1, ps):
            rp.append(r[i:i+ps, j:j+ps].mean()); gp.append((g[i:i+ps, j:j+ps]**2).mean())
    ra, ga = np.array(rp), np.array(gp)
    gb = (ga > np.percentile(ga, 80)).astype(float)
    if gb.sum() == 0 or gb.sum() == len(gb): return float('nan')
    fpr, tpr_a, _ = roc_curve(gb, ra)
    return fpr[min(np.searchsorted(tpr_a, tpr), len(fpr)-1)]

def normalize_01(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-12)

# ---- Test slice generator ----
@torch.no_grad()
def generate_test_slice(h5_path, model, device, acceleration=4, mask_type='random', seed=42, slice_idx=None):
    with h5py.File(h5_path, 'r') as f:
        kspace_full = torch.from_numpy(f['kspace'][()].copy())
        esc_full = torch.from_numpy(f['reconstruction_esc'][()].copy())
        if slice_idx is None: slice_idx = kspace_full.shape[0] // 2
        kspace = kspace_full[slice_idx].to(device)
        esc = esc_full[slice_idx]

    H, W = kspace.shape
    mask = create_mask((H,W), acceleration, mask_type, seed=seed).to(device)
    ifft_c = from_kspace(kspace * mask)
    ifft_crop = center_crop(ifft_c.abs(), (320,320))
    ifft_norm = (ifft_crop - ifft_crop.min()) / (ifft_crop.max() - ifft_crop.min() + 1e-8)

    model.eval()
    unet_out = model(ifft_norm.unsqueeze(0).unsqueeze(0)).squeeze().clamp(0,1)

    esc_dev = esc.to(device)
    esc_norm = (esc_dev - esc_dev.min()) / (esc_dev.max() - esc_dev.min() + 1e-8)
    gt_native = from_kspace(kspace_full[slice_idx].to(device))
    unet_native = lift_unet_to_native(unet_out, esc_dev, ifft_c, gt_native)
    decomp = null_space_decomposition_native(unet_native, gt_native, mask)

    mse = F.mse_loss(unet_out, esc_norm).item()
    return {
        'kspace': kspace.cpu(), 'mask': mask.cpu(), 'ifft_complex': ifft_c.cpu(),
        'ifft_mag_320': ifft_crop.cpu(), 'ifft_norm': ifft_norm.cpu(),
        'unet_out': unet_out.cpu(), 'esc': esc, 'esc_norm': esc_norm.cpu(),
        'null_map': decomp['null_map'].cpu(), 'null_ratio': decomp['null_ratio'],
        'psnr': -10*np.log10(mse+1e-12), 'H': H, 'W': W,
        'h5_path': h5_path, 'slice_idx': slice_idx,
    }

print("All utilities loaded")

## 2. NB04 Detector Functions

All physics-informed detectors from Notebook 04 re-implemented as standalone functions.

In [ ]:
def det_ifft_gradient(ifft_mag):
    img = ifft_mag.cpu().numpy().squeeze() if isinstance(ifft_mag, torch.Tensor) else ifft_mag.squeeze()
    sx, sy = sobel(img, 0), sobel(img, 1)
    grad = np.sqrt(sx**2 + sy**2)
    lv = np.maximum(uniform_filter(img**2, 5) - uniform_filter(img, 5)**2, 0)
    return 0.5 * normalize_01(grad) + 0.5 * normalize_01(lv)

def det_residual(unet, ifft):
    u = unet.cpu().numpy().squeeze() if isinstance(unet, torch.Tensor) else unet.squeeze()
    i = ifft.cpu().numpy().squeeze() if isinstance(ifft, torch.Tensor) else ifft.squeeze()
    return np.abs(u * (i.max()-i.min()) + i.min() - i)

def det_data_consistency(unet, ifft_mag, ifft_c, mask, device):
    un = lift_unet_to_native_inference(unet.to(device), ifft_mag.to(device), ifft_c.to(device))
    m = mask.to(device).squeeze()
    if m.dim() == 1: m = m.unsqueeze(0).expand(un.shape[-2], -1)
    proj = from_kspace(to_kspace(un) * m)
    return center_crop((un - proj).abs(), (320,320)).cpu().numpy().squeeze()

def det_high_freq_energy(unet, ifft):
    u = unet.cpu().numpy().squeeze() if isinstance(unet, torch.Tensor) else unet.squeeze()
    i = ifft.cpu().numpy().squeeze() if isinstance(ifft, torch.Tensor) else ifft.squeeze()
    res = np.abs(u * (i.max()-i.min()) + i.min() - i)
    return 0.5 * normalize_01(np.abs(laplace(res))) + 0.5 * normalize_01(
        np.abs(gaussian_filter(res, 1) - gaussian_filter(res, 3)))

def det_kspace_consistency(unet, kspace, ifft_mag, ifft_c, mask, device):
    un = lift_unet_to_native_inference(unet.to(device), ifft_mag.to(device), ifft_c.to(device))
    m = mask.to(device).squeeze()
    if m.dim() == 1: m = m.unsqueeze(0).expand(un.shape[-2], -1)
    err = from_kspace((to_kspace(un) - kspace.to(device)) * m).abs()
    return center_crop(err, (320,320)).cpu().numpy().squeeze()

def det_learned_risk(ifft_norm, mask, risk_model, device):
    psf_2d = make_psf_2d(mask, 320).to(device)
    inp = torch.stack([ifft_norm.squeeze().to(device), psf_2d], 0).unsqueeze(0)
    risk_model.eval()
    with torch.no_grad(): return risk_model(inp).squeeze().cpu().numpy()

def det_multi_mask(kspace, model, device, acceleration=4, n_masks=8, base_seed=1000):
    H, W = kspace.shape
    outs = []
    for i in range(n_masks):
        m = create_mask((H,W), acceleration, 'random', seed=base_seed+i).to(device)
        ifft = center_crop(from_kspace(kspace.to(device) * m).abs(), (320,320))
        ifft_n = (ifft - ifft.min()) / (ifft.max() - ifft.min() + 1e-8)
        with torch.no_grad(): outs.append(model(ifft_n.unsqueeze(0).unsqueeze(0)).squeeze().cpu().numpy())
    return np.stack(outs).std(axis=0)

def det_reffree_sfrc(unet, ifft_mag, patch_size=48, stride=16, thr=0.75):
    u = unet.cpu().numpy().squeeze() if isinstance(unet, torch.Tensor) else unet.squeeze()
    i = ifft_mag.cpu().numpy().squeeze() if isinstance(ifft_mag, torch.Tensor) else ifft_mag.squeeze()
    u_d = u * (i.max()-i.min()) + i.min()
    risk, count = np.zeros((320,320), np.float32), np.zeros((320,320), np.float32)
    for r in range(0, 320-patch_size+1, stride):
        for c in range(0, 320-patch_size+1, stride):
            p1, p2 = i[r:r+patch_size, c:c+patch_size], u_d[r:r+patch_size, c:c+patch_size]
            win = np.outer(np.hanning(patch_size), np.hanning(patch_size))
            f1, f2 = np.fft.fft2(p1*win), np.fft.fft2(p2*win)
            y, x = np.ogrid[:patch_size, :patch_size]
            rad = np.sqrt((y-patch_size//2)**2 + (x-patch_size//2)**2).astype(int)
            frc = []
            for ri in range(patch_size//4, patch_size//2):
                ring = rad == ri
                if ring.sum() == 0: continue
                num = np.abs(np.sum(f1[ring]*np.conj(f2[ring])))
                den = np.sqrt(np.sum(np.abs(f1[ring])**2)*np.sum(np.abs(f2[ring])**2))
                frc.append(num/(den+1e-12))
            if frc:
                score = np.mean([f < thr for f in frc])
                risk[r:r+patch_size, c:c+patch_size] += score
                count[r:r+patch_size, c:c+patch_size] += 1
    count[count==0] = 1
    return risk / count

print("NB04 detectors ready")

## 3. Generate Test Set

In [ ]:
# Check and re-sync if needed
n_local = len(glob.glob(os.path.join(val_dst, '*.h5')))
if n_local < 100:
    print(f"Only {n_local} files, re-syncing...")
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    subprocess.run(['rsync', '-a', '/content/drive/MyDrive/fastmri/singlecoil_val/', val_dst], capture_output=True)
    n_local = len(glob.glob(os.path.join(val_dst, '*.h5')))
print(f"Val volumes: {n_local}")

val_files = sorted(glob.glob(os.path.join(val_dst, '*.h5')))
test_files = val_files[:100]

test_data = []
t0 = time.time()
for i, h5 in enumerate(tqdm(test_files, desc="Test set")):
    try:
        test_data.append(generate_test_slice(h5, model, device, acceleration=4, seed=42))
    except: pass

print(f"{len(test_data)} slices in {time.time()-t0:.0f}s")
print(f"PSNR: {np.mean([d['psnr'] for d in test_data]):.1f} +/- {np.std([d['psnr'] for d in test_data]):.1f}")
print(f"Null ratio: {np.mean([d['null_ratio'] for d in test_data]):.3f}")

## 4. NB04 Physics Detectors

Compute all 10 physics-informed detectors from NB04 on the test set.

In [ ]:
nb04_names = ['ifft_gradient', 'residual', 'data_consistency', 'kspace_consistency',
              'high_freq_energy', 'learned_risk', 'multi_mask', 'reffree_sfrc',
              'mm_learned_70_30', 'mm_ifft_grad_60_40']

det_maps = {n: [] for n in nb04_names}

for d in tqdm(test_data, desc="NB04 detectors"):
    det_maps['ifft_gradient'].append(det_ifft_gradient(d['ifft_mag_320']))
    det_maps['residual'].append(det_residual(d['unet_out'], d['ifft_mag_320']))
    det_maps['data_consistency'].append(det_data_consistency(d['unet_out'], d['ifft_mag_320'], d['ifft_complex'], d['mask'], device))
    det_maps['kspace_consistency'].append(det_kspace_consistency(d['unet_out'], d['kspace'], d['ifft_mag_320'], d['ifft_complex'], d['mask'], device))
    det_maps['high_freq_energy'].append(det_high_freq_energy(d['unet_out'], d['ifft_mag_320']))
    det_maps['learned_risk'].append(det_learned_risk(d['ifft_norm'], d['mask'], risk_model, device))
    mm = det_multi_mask(d['kspace'], model, device)
    det_maps['multi_mask'].append(mm)
    det_maps['reffree_sfrc'].append(det_reffree_sfrc(d['unet_out'], d['ifft_mag_320']))
    det_maps['mm_learned_70_30'].append(0.7*normalize_01(mm) + 0.3*normalize_01(det_maps['learned_risk'][-1]))
    det_maps['mm_ifft_grad_60_40'].append(0.6*normalize_01(mm) + 0.4*normalize_01(det_maps['ifft_gradient'][-1]))

print("\nNB04 results:")
for name in nb04_names:
    aurocs = [compute_auroc_patches(det_maps[name][i], test_data[i]['null_map'], 32) for i in range(len(test_data))]
    aurocs = [a for a in aurocs if not np.isnan(a)]
    print(f"  {name:<25}: {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

## 5. MC Dropout

The U-Net has `Dropout2d(p=0.05)` baked in. At inference, calling `model.train()` activates dropout while keeping BatchNorm in eval mode. 20 stochastic forward passes, per-pixel standard deviation as uncertainty.

Note: p=0.05 is low. If variance is too small, we also test MC Feature Dropout at the bottleneck with p=0.5 (no retraining needed, injected via forward hook).

In [ ]:
def det_mc_dropout(ifft_norm, model, device, n_samples=20):
    model.train()
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)): m.eval()
    inp = ifft_norm.unsqueeze(0).unsqueeze(0).to(device) if ifft_norm.dim() == 2 else ifft_norm.unsqueeze(0).to(device)
    if inp.dim() == 3: inp = inp.unsqueeze(0)
    outs = []
    with torch.no_grad():
        for _ in range(n_samples): outs.append(model(inp).squeeze().cpu().numpy())
    model.eval()
    return np.stack(outs).std(axis=0)

det_maps['mc_dropout'] = []
for d in tqdm(test_data, desc="MC Dropout"):
    det_maps['mc_dropout'].append(det_mc_dropout(d['ifft_norm'], model, device, 20))
model.eval()

aurocs = [compute_auroc_patches(det_maps['mc_dropout'][i], test_data[i]['null_map'], 32) for i in range(len(test_data))]
aurocs = [a for a in aurocs if not np.isnan(a)]
print(f"MC Dropout (p=0.05): {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")
print(f"  Mean std: {np.mean([det_maps['mc_dropout'][i].mean() for i in range(len(test_data))]):.6f}")

## 6. Energy OOD, Mahalanobis Distance, TTA, Saliency, Lipschitz

Five additional detector families. Energy OOD and Mahalanobis are classification-to-regression transfers that may not work well for diffuse hallucinations. TTA flip exploits the geometric symmetry of MRI magnitude images.

In [ ]:
# ---- Energy OOD ----
def det_energy_ood(ifft_norm, model, device, T=1.0):
    features = {}
    def hook(name):
        def fn(m, i, o): features[name] = o.detach()
        return fn
    h1 = model.bottleneck.register_forward_hook(hook('bottleneck'))
    model.eval()
    inp = ifft_norm.unsqueeze(0).unsqueeze(0).to(device) if ifft_norm.dim()==2 else ifft_norm.unsqueeze(0).to(device)
    if inp.dim()==3: inp=inp.unsqueeze(0)
    with torch.no_grad(): _ = model(inp)
    h1.remove()
    e = -T * torch.logsumexp(features['bottleneck']/T, dim=1)
    return F.interpolate(e.unsqueeze(1), (320,320), mode='bilinear', align_corners=False).squeeze().cpu().numpy()

det_maps['energy_bottleneck'] = []
for d in tqdm(test_data, desc="Energy OOD"):
    det_maps['energy_bottleneck'].append(det_energy_ood(d['ifft_norm'], model, device))

# ---- Mahalanobis ----
print("\nFitting Mahalanobis on training features...")
train_src = '/content/drive/MyDrive/fastmri/singlecoil_train/'
train_sample = sorted(glob.glob(os.path.join(train_src, '*.h5')))[:20]

hook_store = {}
def hook_bn(m, i, o): hook_store['feat'] = o.detach()
h_bn = model.bottleneck.register_forward_hook(hook_bn)
model.eval()

feats = []
for h5 in tqdm(train_sample, desc="Train features"):
    try:
        with h5py.File(h5, 'r') as f:
            ks = torch.from_numpy(f['kspace'][f['kspace'].shape[0]//2].copy()).to(device)
        H, W = ks.shape
        mask = create_mask((H,W), 4, 'random', seed=42).to(device)
        ifft = center_crop(from_kspace(ks*mask).abs(), (320,320))
        ifft_n = (ifft-ifft.min())/(ifft.max()-ifft.min()+1e-8)
        with torch.no_grad(): _ = model(ifft_n.unsqueeze(0).unsqueeze(0))
        feats.append(hook_store['feat'].squeeze(0).cpu())
    except: pass
h_bn.remove()

feats_t = torch.stack(feats)
N, C, Hf, Wf = feats_t.shape
feats_flat = feats_t.permute(0,2,3,1).reshape(-1, C).numpy()
feat_mean = feats_flat.mean(0)
cov = np.cov((feats_flat-feat_mean).T) + 1e-5*np.eye(C)
cov_inv = np.linalg.inv(cov)
print(f"Mahalanobis: {N} train slices, C={C}, cond={np.linalg.cond(cov):.1e}")

det_maps['mahalanobis'] = []
h_bn2 = model.bottleneck.register_forward_hook(hook_bn)
for d in tqdm(test_data, desc="Mahalanobis"):
    inp = d['ifft_norm'].unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad(): _ = model(inp)
    feat = hook_store['feat'].squeeze(0).cpu().numpy()
    diff = feat.transpose(1,2,0).reshape(-1, C) - feat_mean
    mah = np.sqrt(np.sum(diff @ cov_inv * diff, 1)).reshape(feat.shape[1], feat.shape[2])
    det_maps['mahalanobis'].append(F.interpolate(
        torch.from_numpy(mah).float().unsqueeze(0).unsqueeze(0), (320,320),
        mode='bilinear', align_corners=False).squeeze().numpy())
h_bn2.remove()

# ---- TTA Flip ----
det_maps['tta_flip'] = []
det_maps['tta_flip_rot'] = []
model.eval()
for d in tqdm(test_data, desc="TTA"):
    inp = d['ifft_norm'].unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        o0 = model(inp).squeeze().cpu().numpy()
        o_h = torch.flip(model(torch.flip(inp, [-1])), [-1]).squeeze().cpu().numpy()
        o_v = torch.flip(model(torch.flip(inp, [-2])), [-2]).squeeze().cpu().numpy()
        o_r = torch.flip(model(torch.flip(inp, [-1,-2])), [-1,-2]).squeeze().cpu().numpy()
    det_maps['tta_flip'].append(np.abs(o0 - o_h))
    det_maps['tta_flip_rot'].append(np.stack([o0, o_h, o_v, o_r]).std(0))

# ---- Input Gradient ----
det_maps['input_gradient'] = []
model.eval()
for d in tqdm(test_data, desc="Input gradient"):
    inp = d['ifft_norm'].unsqueeze(0).unsqueeze(0).to(device)
    inp.requires_grad_(True)
    model(inp).sum().backward()
    det_maps['input_gradient'].append(inp.grad.squeeze().abs().cpu().numpy())
    inp.requires_grad_(False)

# ---- Local Lipschitz ----
det_maps['local_lipschitz'] = []
model.eval()
for d in tqdm(test_data, desc="Lipschitz"):
    inp = d['ifft_norm'].unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        o0 = model(inp).squeeze().cpu().numpy()
        outs = [model(inp + torch.randn_like(inp)*0.01).squeeze().cpu().numpy() for _ in range(5)]
    det_maps['local_lipschitz'].append(np.max(np.abs(np.stack(outs) - o0), 0) / 0.01)

# ---- sFRC with GT ----
def det_sfrc_gt(unet, esc_norm, ps=48, stride=16, thr=0.75):
    u = unet.cpu().numpy().squeeze() if isinstance(unet, torch.Tensor) else unet.squeeze()
    g = esc_norm.cpu().numpy().squeeze() if isinstance(esc_norm, torch.Tensor) else esc_norm.squeeze()
    risk, count = np.zeros((320,320), np.float32), np.zeros((320,320), np.float32)
    for r in range(0, 320-ps+1, stride):
        for c in range(0, 320-ps+1, stride):
            win = np.outer(np.hanning(ps), np.hanning(ps))
            f1, f2 = np.fft.fft2(u[r:r+ps,c:c+ps]*win), np.fft.fft2(g[r:r+ps,c:c+ps]*win)
            y, x = np.ogrid[:ps,:ps]
            rad = np.sqrt((y-ps//2)**2+(x-ps//2)**2).astype(int)
            frc = []
            for ri in range(ps//4, ps//2):
                ring = rad==ri
                if ring.sum()==0: continue
                num = np.abs(np.sum(f1[ring]*np.conj(f2[ring])))
                den = np.sqrt(np.sum(np.abs(f1[ring])**2)*np.sum(np.abs(f2[ring])**2))
                frc.append(num/(den+1e-12))
            if frc: score = np.mean([f<thr for f in frc])
            else: score = 0
            risk[r:r+ps,c:c+ps] += score; count[r:r+ps,c:c+ps] += 1
    count[count==0]=1
    return risk/count

det_maps['sfrc_gt'] = []
for d in tqdm(test_data, desc="sFRC GT"):
    det_maps['sfrc_gt'].append(det_sfrc_gt(d['unet_out'], d['esc_norm']))

# Quick results
print("\n--- NB05 detector results ---")
for name in ['mc_dropout', 'energy_bottleneck', 'mahalanobis', 'tta_flip', 'tta_flip_rot',
             'input_gradient', 'local_lipschitz', 'sfrc_gt']:
    aurocs = [compute_auroc_patches(det_maps[name][i], test_data[i]['null_map'], 32)
              for i in range(len(test_data))]
    aurocs = [a for a in aurocs if not np.isnan(a)]
    print(f"  {name:<25}: {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

## 7. Deep Ensemble Training

Three U-Nets trained with different random seeds (100, 200, 300). Same architecture, same hyperparameters. Per-pixel standard deviation across the three outputs captures epistemic uncertainty about the null-space content.

In [ ]:
# Sync train data locally
train_dst = '/content/data/singlecoil_train/'
os.makedirs(train_dst, exist_ok=True)
train_drive = sorted(glob.glob('/content/drive/MyDrive/fastmri/singlecoil_train/*.h5'))[:150]
for f in tqdm(train_drive, desc="Syncing train"):
    dst = os.path.join(train_dst, os.path.basename(f))
    if not os.path.exists(dst): os.system(f'cp "{f}" "{dst}"')
print(f"Local train: {len(glob.glob(os.path.join(train_dst, '*.h5')))} volumes")

def train_ensemble_member(seed, train_root, val_root, device, n_epochs=50, batch_size=8):
    print(f"\n--- Seed {seed} ---")
    torch.manual_seed(seed); np.random.seed(seed)
    m = UNet(channels=(32,64,128,256), dropout_p=0.05).to(device)
    train_ds = FastMRIDataset(train_root, fixed_masks=False, seed=seed, augment=True)
    val_ds = FastMRIDataset(val_root, fixed_masks=True, seed=seed)
    train_ld = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_ld = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-5)
    scaler = torch.amp.GradScaler('cuda')
    best_val = float('inf')

    for epoch in range(1, n_epochs+1):
        m.train(); tl, nb = 0, 0
        for inp, tgt in train_ld:
            inp, tgt = inp.to(device), tgt.to(device)
            opt.zero_grad()
            with torch.amp.autocast('cuda'): loss = F.l1_loss(m(inp), tgt)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item(); nb += 1
        m.eval(); vl, nv = 0, 0
        with torch.no_grad():
            for inp, tgt in val_ld:
                inp, tgt = inp.to(device), tgt.to(device)
                with torch.amp.autocast('cuda'): vl += F.l1_loss(m(inp), tgt).item(); nv += 1
        sched.step()
        vl /= max(nv, 1)
        if vl < best_val:
            best_val = vl
            torch.save({'model_state_dict': m.state_dict(), 'epoch': epoch, 'val_loss': vl, 'seed': seed},
                       os.path.join(ckpt_dir, f'unet_ensemble_seed{seed}.pt'))
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Ep {epoch}: train={tl/nb:.4f}, val={vl:.4f}, best={best_val:.4f}")
    print(f"  Done seed={seed}, best val L1={best_val:.4f}")

t0 = time.time()
for seed in [100, 200, 300]:
    train_ensemble_member(seed, train_dst, val_dst, device)
print(f"\nEnsemble training: {(time.time()-t0)/3600:.1f} hours")

In [ ]:
import os
local_ckpt = '/content/checkpoints/'
ckpt_dir = '/content/drive/MyDrive/fastmri/checkpoints/'

from google.colab import drive
drive.mount('/content/drive')

for seed in [100, 200, 300]:
    src = os.path.join(local_ckpt, f'unet_ensemble_seed{seed}.pt')
    if os.path.exists(src):
        !cp "{src}" "{ckpt_dir}"
        print(f"Copied seed {seed}")
print("Done")# Copy local checkpoints to Drive (run when wifi is back)
for seed in [100, 200, 300]:
    src = os.path.join(local_ckpt, f'unet_ensemble_seed{seed}.pt')
    if os.path.exists(src):
        !cp "{src}" "{ckpt_dir}"
        print(f"Copied seed {seed}")
print("Done")

## 8. Deep Ensemble Evaluation

In [ ]:
ensemble_models = []
for seed in [100, 200, 300]:
    m = UNet(channels=(32,64,128,256), dropout_p=0.05).to(device)
    ckpt = torch.load(os.path.join(ckpt_dir, f'unet_ensemble_seed{seed}.pt'), map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state_dict']); m.eval()
    print(f"Loaded seed={seed}, val L1={ckpt['val_loss']:.4f}, epoch={ckpt['epoch']}")
    ensemble_models.append(m)

det_maps['deep_ensemble'] = []
for d in tqdm(test_data, desc="Deep Ensemble"):
    inp = d['ifft_norm'].unsqueeze(0).unsqueeze(0).to(device)
    outs = []
    for m in ensemble_models:
        m.eval()
        with torch.no_grad(): outs.append(m(inp).squeeze().cpu().numpy())
    det_maps['deep_ensemble'].append(np.stack(outs).std(0))

aurocs = [compute_auroc_patches(det_maps['deep_ensemble'][i], test_data[i]['null_map'], 32) for i in range(len(test_data))]
aurocs = [a for a in aurocs if not np.isnan(a)]
print(f"Deep Ensemble (3): {np.mean(aurocs):.3f} +/- {np.std(aurocs):.3f}")

## 9. Full Comparison Table

In [ ]:
family_map = {
    'ifft_gradient': 'Physics', 'residual': 'Physics', 'data_consistency': 'Physics',
    'kspace_consistency': 'Physics', 'high_freq_energy': 'Physics',
    'learned_risk': 'Physics', 'multi_mask': 'Physics', 'reffree_sfrc': 'Physics',
    'mm_learned_70_30': 'Phys.combo', 'mm_ifft_grad_60_40': 'Phys.combo',
    'mc_dropout': 'Uncertainty', 'deep_ensemble': 'Uncertainty',
    'energy_bottleneck': 'OOD', 'mahalanobis': 'OOD',
    'tta_flip': 'Geometric', 'tta_flip_rot': 'Geometric',
    'input_gradient': 'Attribution', 'local_lipschitz': 'Stability',
    'sfrc_gt': 'Freq.fidel.',
}

available = [n for n in det_maps if len(det_maps[n]) == len(test_data)]
full_results = {}

print(f"{'Detector':<25} | {'Family':<12} | {'AUROC 32':>8} | {'±':>5} | {'FPR@95':>7}")
print("-" * 70)

for name in available:
    a32s, f95s = [], []
    for i, d in enumerate(test_data):
        a = compute_auroc_patches(det_maps[name][i], d['null_map'], 32)
        f = compute_fpr_at_tpr_patches(det_maps[name][i], d['null_map'], 0.95, 32)
        if not np.isnan(a): a32s.append(a)
        if not np.isnan(f): f95s.append(f)
    full_results[name] = {'auroc_32': np.mean(a32s), 'auroc_32_std': np.std(a32s),
                           'fpr95': np.mean(f95s), 'fpr95_std': np.std(f95s)}
    fam = family_map.get(name, '?')
    print(f"{name:<25} | {fam:<12} | {np.mean(a32s):8.3f} | {np.std(a32s):5.3f} | {np.mean(f95s):7.3f}")

# Sorted
print(f"\n--- Sorted by AUROC 32x32 ---")
for rank, name in enumerate(sorted(available, key=lambda n: full_results[n]['auroc_32'], reverse=True), 1):
    r = full_results[name]
    print(f"  {rank:2d}. {name:<25} | {family_map.get(name,'?'):<12} | {r['auroc_32']:.3f} +/- {r['auroc_32_std']:.3f}")

# Combinations
print(f"\n--- Best combinations (32x32) ---")
top = sorted(available, key=lambda n: full_results[n]['auroc_32'], reverse=True)[:6]
top = [n for n in top if n not in ['mm_learned_70_30', 'mm_ifft_grad_60_40']]

from itertools import combinations
for size in [2, 3, 4]:
    best_score, best_combo = 0, None
    for combo in combinations(top[:5], size):
        aurocs = []
        for i, d in enumerate(test_data):
            c = sum(normalize_01(det_maps[n][i]) for n in combo) / len(combo)
            a = compute_auroc_patches(c, d['null_map'], 32)
            if not np.isnan(a): aurocs.append(a)
        score = np.mean(aurocs) if aurocs else 0
        if score > best_score:
            best_score, best_combo = score, combo
    print(f"  Best {size}-way: {' + '.join(n[:12] for n in best_combo)} = {best_score:.3f}")

## 10. Save Results

In [ ]:
os.makedirs('figures', exist_ok=True)

# ---- Figure 1: Detector ranking bar chart ----
sorted_names = sorted(available, key=lambda n: full_results[n]['auroc_32'], reverse=True)
colors_bar = {'Physics': '#2196F3', 'Phys.combo': '#1565C0', 'Uncertainty': '#FF9800',
              'OOD': '#F44336', 'Geometric': '#4CAF50', 'Attribution': '#9C27B0',
              'Stability': '#795548', 'Freq.fidel.': '#607D8B'}

fig, ax = plt.subplots(figsize=(12, 6))
y_pos = range(len(sorted_names))
bars = ax.barh(y_pos,
    [full_results[n]['auroc_32'] for n in sorted_names],
    xerr=[full_results[n]['auroc_32_std'] for n in sorted_names],
    capsize=3, color=[colors_bar.get(family_map.get(n, '?'), '#999') for n in sorted_names],
    edgecolor='black', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('AUROC (32x32 patches)', fontsize=12)
ax.set_title('Hallucination Detector Ranking', fontsize=14)
ax.axvline(x=0.5, color='gray', ls='--', alpha=0.5, label='Random')
ax.set_xlim(0.3, 1.0)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

# Legend for families
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f) for f, c in colors_bar.items() if f in set(family_map.get(n,'?') for n in sorted_names)]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('figures/05_detector_ranking.png', bbox_inches='tight')
plt.show()


# ---- Figure 2: Detection maps for one example ----
ex = test_data[0]
det_show = ['ifft_gradient', 'multi_mask', 'tta_flip', 'deep_ensemble', 'learned_risk']
det_show = [n for n in det_show if n in det_maps and len(det_maps[n]) > 0]

fig, axes = plt.subplots(2, len(det_show) + 2, figsize=(4*(len(det_show)+2), 8))

# Row 1: images
axes[0,0].imshow(ex['esc'].numpy(), cmap='gray'); axes[0,0].set_title('Ground Truth'); axes[0,0].axis('off')
unet_phys = ex['unet_out'].numpy() * (ex['esc'].max()-ex['esc'].min()).item() + ex['esc'].min().item()
axes[0,1].imshow(unet_phys, cmap='gray'); axes[0,1].set_title(f'U-Net ({ex["psnr"]:.1f} dB)'); axes[0,1].axis('off')
for j, name in enumerate(det_show):
    axes[0,j+2].imshow(det_maps[name][0], cmap='hot')
    axes[0,j+2].set_title(name.replace('_',' '), fontsize=10); axes[0,j+2].axis('off')

# Row 2: null-space GT + overlay
nm = ex['null_map'].numpy()
vmax = np.percentile(nm, 99)
axes[1,0].imshow(nm, cmap='hot', vmin=0, vmax=vmax); axes[1,0].set_title('Hallucination GT'); axes[1,0].axis('off')
axes[1,1].imshow(ex['esc'].numpy(), cmap='gray')
axes[1,1].imshow(nm, cmap='hot', alpha=0.5, vmin=0, vmax=vmax)
axes[1,1].set_title('GT on Anatomy'); axes[1,1].axis('off')
for j, name in enumerate(det_show):
    rm = det_maps[name][0]
    rm_n = normalize_01(rm)
    axes[1,j+2].imshow(ex['esc'].numpy(), cmap='gray')
    axes[1,j+2].imshow(rm_n, cmap='hot', alpha=0.5)
    a = compute_auroc_patches(rm, ex['null_map'], 32)
    axes[1,j+2].set_title(f'AUROC={a:.3f}', fontsize=10); axes[1,j+2].axis('off')

plt.suptitle('Detection Maps: Ground Truth vs Top Detectors', fontsize=14)
plt.tight_layout()
plt.savefig('figures/05_detection_maps.png', bbox_inches='tight')
plt.show()


# ---- Figure 3: Correlation heatmap (top detectors) ----
corr_names = [n for n in sorted_names[:10] if n in det_maps]
n_c = len(corr_names)
corr_mat = np.zeros((n_c, n_c))
n_s = min(50, len(test_data))

for i_s in range(n_s):
    vecs = []
    for name in corr_names:
        rm = det_maps[name][i_s]
        rm = rm.detach().cpu().numpy().squeeze() if isinstance(rm, torch.Tensor) else np.asarray(rm).squeeze()
        patches = []
        for pi in range(0, 320-32+1, 32):
            for pj in range(0, 320-32+1, 32):
                patches.append(rm[pi:pi+32, pj:pj+32].mean())
        vecs.append(np.array(patches))
    for i in range(n_c):
        for j in range(n_c):
            corr_mat[i,j] += np.corrcoef(vecs[i], vecs[j])[0,1]
corr_mat /= n_s

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n_c)); ax.set_yticks(range(n_c))
short_names = [n[:12] for n in corr_names]
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)
for i in range(n_c):
    for j in range(n_c):
        ax.text(j, i, f'{corr_mat[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(corr_mat[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046, label='Pearson r')
ax.set_title('Detector Complementarity (patch-level correlation)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/05_correlation_heatmap.png', bbox_inches='tight')
plt.show()

!cp figures/05_*.png /content/drive/MyDrive/fastmri/checkpoints/ 2>/dev/null || true
print("Figures saved")

![Figure 1](../figures/figures_notebook_5/fig_001.png?raw=1)


![Figure 2](../figures/figures_notebook_5/fig_002.png?raw=1)


![Figure 3](../figures/figures_notebook_5/fig_003.png?raw=1)


In [ ]:
save_dict = {
    'n_test_slices': len(test_data),
    'test_psnr_mean': float(np.mean([d['psnr'] for d in test_data])),
    'test_null_ratio_mean': float(np.mean([d['null_ratio'] for d in test_data])),
    'ranking': {
        'deep_ensemble': 0.859, 'ifft_gradient': 0.828,
        'mm_ifft_grad_60_40': 0.804, 'tta_flip': 0.801,
        'tta_flip_rot': 0.795, 'high_freq_energy': 0.782,
        'multi_mask': 0.776, 'mahalanobis': 0.686,
        'mc_dropout': 0.653, 'energy_bottleneck': 0.576,
        'sfrc_gt': 0.432,
    },
    'best_combination': {'deep_ensemble + high_freq_energy': 0.869},
    'ensemble_training': {
        'seed_100': {'val_l1': 0.0430, 'epoch': 2},
        'seed_200': {'val_l1': 0.0557, 'epoch': 43},
        'seed_300': {'val_l1': 0.0547, 'epoch': 35},
    },
}

save_path = os.path.join(ckpt_dir, 'nb05_results_final.json')
with open(save_path, 'w') as f:
    json.dump(save_dict, f, indent=2)
!cp figures/05_*.png /content/drive/MyDrive/fastmri/checkpoints/ 2>/dev/null || true
print(f"Saved to {save_path}")

## Summary

| Rank | Detector | Family | AUROC (32x32) | GT-free? |
|------|----------|--------|--------------|----------|
| 1 | Deep Ensemble (3) | Uncertainty | 0.859 ± 0.081 | Yes |
| 2 | IFFT Gradient | Physics | 0.828 ± 0.065 | Yes |
| 3 | MM + IFFT Grad (60/40) | Phys.combo | 0.804 ± 0.078 | Yes |
| 4 | TTA Flip | Geometric | 0.801 ± 0.085 | Yes |
| 5 | TTA Flip+Rot | Geometric | 0.795 ± 0.086 | Yes |
| 6 | High-Freq Energy | Physics | 0.782 ± 0.091 | Yes |
| 7 | Multi-Mask (8) | Physics | 0.776 ± 0.082 | Yes |
| 8 | Mahalanobis | OOD | 0.686 ± 0.105 | Yes |
| 9 | MC Dropout (p=0.05) | Uncertainty | 0.653 ± 0.117 | Yes |
| 10 | Energy OOD | OOD | 0.576 ± 0.148 | Yes |
| 11 | sFRC GT | Freq.fidel. | 0.432 ± 0.076 | No |

Best combination: Deep Ensemble + High-Freq Energy = 0.869.

**Key findings:**

1. Deep Ensemble dominates. Three models trained with different seeds learn different opinions about missing k-space content. Their disagreement directly reveals hallucinated regions.

2. IFFT Gradient (0.828) remains the best zero-cost detector across all notebooks. No learning, no extra forward passes, just Sobel + local variance on the already-available IFFT image.

3. TTA Flip (0.801) is a novel finding not present in the reference literature. MRI magnitude is flip-symmetric, but the U-Net fabricates different hallucinations for flipped input. Only 2 forward passes, cheaper than multi-mask (8) or MC dropout (20).

4. OOD methods designed for classification fail for diffuse regression hallucinations. Energy OOD (0.576) and sFRC GT (0.432) are near or below chance. These methods assume hallucinations are localized anomalies, but MRI hallucinations are diffuse high-frequency texture spread across the image.

5. MC Dropout at p=0.05 is too weak (0.653). The training dropout rate is too low to produce meaningful variance. Higher dropout injected at the bottleneck (as tested in earlier sessions) works better but requires modifying the forward pass.

6. The combination ceiling of 0.869 reflects a fundamental ambiguity: some hallucinated content is anatomically plausible and consistent across all detection methods.